# Decision Tree — Baseline
Treino em 2020–2023, teste em 2024. Sem SMOTE — desbalanceamento tratado via `class_weight='balanced'`.
Decision Tree não lida com NaN nativamente — imputa-se antes do classificador.
Métricas prioritárias: **Sensibilidade > AUPRC > ROC-AUC > Especificidade > F1**

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.metrics import (
    precision_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

BASE_DIR     = '../../data/features/baseline'
OUTPUT_MOD   = '../../output/modelos'
OUTPUT_MET   = '../../output/metricas'
ALGO         = 'decision_tree'
DATASET      = 'baseline'
RANDOM_STATE = 42
YEAR_COL     = 'year'

## 1. Carregamento dos dados

In [ ]:
X_train = pd.read_parquet(os.path.join(BASE_DIR, 'X_train.parquet'))
y_train = pd.read_parquet(os.path.join(BASE_DIR, 'y_train.parquet')).squeeze()
X_test  = pd.read_parquet(os.path.join(BASE_DIR, 'X_test.parquet'))
y_test  = pd.read_parquet(os.path.join(BASE_DIR, 'y_test.parquet')).squeeze()

mask    = y_train.notna()
X_train = X_train[mask]
y_train = y_train[mask]

print(f'X_train: {X_train.shape} | Óbitos: {int(y_train.sum()):,} ({y_train.mean()*100:.2f}%)')
print(f'X_test:  {X_test.shape}  | Óbitos: {int(y_test.sum()):,} ({y_test.mean()*100:.2f}%)')

## 2. Pipeline
Decision Tree não aceita NaN — imputa-se mediana para contínuas e moda para binárias.

In [ ]:
ALRM_COLS = [c for c in X_train.columns if c.startswith('ALRM_')]
GRAV_COLS = [c for c in X_train.columns if c.startswith('GRAV_')]
SYMP_COLS = [
    'FEBRE','MIALGIA','CEFALEIA','EXANTEMA','VOMITO','NAUSEA',
    'DOR_COSTAS','CONJUNTVIT','ARTRITE','ARTRALGIA','PETEQUIA_N',
    'LEUCOPENIA','LACO','DOR_RETRO','DIABETES','HEMATOLOG',
    'HEPATOPAT','RENAL','HIPERTENSA','AUTO_IMUNE',
]
SYMP_COLS = [c for c in SYMP_COLS if c in X_train.columns]

preprocessor = ColumnTransformer(
    transformers=[
        ('alrm_grav',
         SimpleImputer(strategy='constant', fill_value=0),
         ALRM_COLS + GRAV_COLS),

        ('sintomas',
         SimpleImputer(strategy='most_frequent'),
         SYMP_COLS),

        ('continuas',
         SimpleImputer(strategy='median'),
         ['age_years', 'epi_week']),

        ('sexo',
         Pipeline([
             ('enc', OrdinalEncoder(categories=[['F', 'M']],
                                    handle_unknown='use_encoded_value',
                                    unknown_value=np.nan)),
             ('imp', SimpleImputer(strategy='most_frequent')),
         ]),
         ['CS_SEXO']),

        ('escol',
         SimpleImputer(strategy='median'),
         ['CS_ESCOL_N']),

        ('raca',
         Pipeline([
             ('imp', SimpleImputer(strategy='most_frequent')),
             ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
         ]),
         ['CS_RACA']),

        ('gestant',
         Pipeline([
             ('imp', SimpleImputer(strategy='most_frequent')),
             ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
         ]),
         ['CS_GESTANT']),

        ('uf',
         OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
         ['SG_UF']),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    )),
])

print('Pipeline definido.')

## 3. Treinamento

In [ ]:
def prep_X(df):
    df = df.copy()
    if 'age_years' in df.columns:
        df.loc[df['age_years'] > 120, 'age_years'] = np.nan
    return df.drop(columns=[YEAR_COL], errors='ignore')

X_train_prep = prep_X(X_train)
X_test_prep  = prep_X(X_test)

pipeline.fit(X_train_prep, y_train)
print('Treinamento concluído.')
print(f'Profundidade real da árvore: {pipeline["clf"].get_depth()}')
print(f'Número de folhas: {pipeline["clf"].get_n_leaves()}')

## 4. Avaliação

In [ ]:
y_te  = y_test.dropna()
proba = pipeline.predict_proba(X_test_prep)[:, 1]
proba = proba[y_test.notna().values]

def calcular_metricas(y_true, y_pred_proba, threshold=0.5):
    y_pred = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'sensibilidade':  round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0,
        'especificidade': round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0,
        'auprc':          round(average_precision_score(y_true, y_pred_proba), 4),
        'roc_auc':        round(roc_auc_score(y_true, y_pred_proba), 4),
        'f1':             round(f1_score(y_true, y_pred), 4),
        'precisao':       round(precision_score(y_true, y_pred, zero_division=0), 4),
        'threshold':      threshold,
        'n_train':        len(X_train_prep),
        'n_obito_train':  int(y_train.sum()),
    }

metricas = calcular_metricas(y_te, proba)
print('=== Decision Tree — Baseline (2020–2023 → 2024) ===')
for k, v in metricas.items():
    print(f'  {k}: {v}')

print()
print(classification_report(y_te, (proba >= 0.5).astype(int), target_names=['Cura', 'Óbito']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_te, proba, ax=axes[0], name='Decision Tree')
axes[0].set_title('Curva ROC — teste 2024')
PrecisionRecallDisplay.from_predictions(y_te, proba, ax=axes[1], name='Decision Tree')
axes[1].set_title('Curva Precision-Recall — teste 2024')
plt.tight_layout()
plt.show()

## 5. Matriz de Confusão (threshold = 0.5)

In [ ]:
y_pred_05      = (proba >= 0.5).astype(int)
cm             = confusion_matrix(y_te, y_pred_05)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay(cm, display_labels=['Cura', 'Óbito']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — contagens')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=['Cura', 'Óbito']).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Matriz de Confusão — normalizada (recall por classe)')
for text in axes[1].texts:
    text.set_text(f'{float(text.get_text()):.2%}')

plt.tight_layout()
plt.show()

print(f'VP (óbitos detectados):   {tp:,}')
print(f'FN (óbitos perdidos):     {fn:,}')
print(f'FP (falsos alarmes):      {fp:,}')
print(f'VN (curas classificadas): {tn:,}')

## 7. Importância das features

In [ ]:
FEATURE_LABELS = {
    # Variáveis contínuas / temporais
    'age_years':          'Idade',
    'epi_week':           'Semana epidemiológica',
    # Dados do paciente
    'CS_SEXO':            'Sexo',
    'CS_GESTANT':         'Gestante',
    'CS_RACA':            'Raça/Cor',
    'CS_ESCOL_N':         'Escolaridade',
    'SG_UF':              'UF de residência',
    # CS_RACA one-hot
    'CS_RACA_1.0':        'Raça: Branca',
    'CS_RACA_2.0':        'Raça: Preta',
    'CS_RACA_3.0':        'Raça: Amarela',
    'CS_RACA_4.0':        'Raça: Parda',
    'CS_RACA_5.0':        'Raça: Indígena',
    # CS_GESTANT one-hot
    'CS_GESTANT_1.0':     'Gestante: 1º trimestre',
    'CS_GESTANT_2.0':     'Gestante: 2º trimestre',
    'CS_GESTANT_3.0':     'Gestante: 3º trimestre',
    'CS_GESTANT_4.0':     'Gestante: idade gestacional ignorada',
    'CS_GESTANT_5.0':     'Gestante: não',
    'CS_GESTANT_6.0':     'Gestante: não se aplica',
    # Sintomas clássicos
    'FEBRE':              'Febre',
    'MIALGIA':            'Mialgia',
    'CEFALEIA':           'Cefaleia',
    'EXANTEMA':           'Exantema',
    'VOMITO':             'Vômito',
    'NAUSEA':             'Náusea',
    'DOR_COSTAS':         'Dor nas costas',
    'CONJUNTVIT':         'Conjuntivite',
    'ARTRITE':            'Artrite',
    'ARTRALGIA':          'Artralgia',
    'PETEQUIA_N':         'Petéquia',
    'LEUCOPENIA':         'Leucopenia',
    'LACO':               'Prova do laço',
    'DOR_RETRO':          'Dor retro-orbital',
    # Comorbidades
    'DIABETES':           'Diabetes',
    'HEMATOLOG':          'Doença hematológica',
    'HEPATOPAT':          'Hepatopatia',
    'RENAL':              'Doença renal',
    'HIPERTENSA':         'Hipertensão arterial',
    'AUTO_IMUNE':         'Doença autoimune',
    # Sinais de alarme
    'ALRM_ABDOM':         'Dor abdominal intensa',
    'ALRM_VOM':           'Vômitos persistentes',
    'ALRM_LIQ':           'Acúmulo de líquidos',
    'ALRM_HEMAT':         'Aumento do hematócrito',
    'ALRM_PLAQ':          'Queda abrupta de plaquetas',
    'ALRM_SANG':          'Sangramento de mucosa',
    'ALRM_LETAR':         'Letargia/irritabilidade',
    'ALRM_HEPAT':         'Hepatomegalia',
    'ALRM_HIPOT':         'Hipotensão postural',
    # Dengue grave
    'GRAV_HIPOT':         'Hipotensão arterial',
    'GRAV_PULSO':         'Pulso fraco e filiforme',
    'GRAV_EXTRE':         'Extremidades frias',
    'GRAV_ENCH':          'Enchimento capilar lento',
    'GRAV_TAQUI':         'Taquicardia',
    'GRAV_CONV':          'Convulsão',
    'GRAV_INSUF':         'Insuficiência respiratória',
    'GRAV_HEMAT':         'Hematêmese',
    'GRAV_MELEN':         'Melena',
    'GRAV_METRO':         'Metrorragia',
    'GRAV_SANG':          'Sangramento intenso',
    'GRAV_AST':           'Astenia grave',
    'GRAV_MIOC':          'Miocardite',
    'GRAV_CONSC':         'Alteração da consciência',
    'GRAV_ORGAO':         'Disfunção orgânica',
}

feature_names = pipeline['pre'].get_feature_names_out()
importances   = pipeline['clf'].feature_importances_

df_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
df_imp['label'] = df_imp['feature'].map(lambda f: FEATURE_LABELS.get(f, f))
df_imp = df_imp.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
top_n  = 10
df_top = df_imp.head(top_n)
ax.barh(df_top['label'][::-1], df_top['importance'][::-1], color='#4C72B0')
ax.set_title(f'Top {top_n} features — Decision Tree (importância por impureza)')
ax.set_xlabel('Importância')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

display(df_imp.head(15)[['label', 'feature', 'importance']])

## 8. Visualização da árvore (ramos superiores)
Exibe apenas os primeiros níveis para interpretação clínica.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    pipeline['clf'],
    feature_names=pipeline['pre'].get_feature_names_out(),
    class_names=['Cura', 'Óbito'],
    filled=True,
    rounded=True,
    max_depth=3,
    ax=ax,
    fontsize=9,
)
ax.set_title('Decision Tree — primeiros 3 níveis', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('../../output/plots', exist_ok=True)
plt.savefig(os.path.join('../../output/plots', 'decision_tree_structure.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Salvamento

In [ ]:
os.makedirs(OUTPUT_MOD, exist_ok=True)
os.makedirs(OUTPUT_MET, exist_ok=True)

model_path = os.path.join(OUTPUT_MOD, f'{ALGO}_{DATASET}.joblib')
joblib.dump(pipeline, model_path)
print(f'Modelo salvo: {model_path}')

df_met = pd.DataFrame([metricas])
df_met['label']   = f'{ALGO}_{DATASET}'
df_met['dataset'] = DATASET
met_path = os.path.join(OUTPUT_MET, f'{ALGO}_{DATASET}.parquet')
df_met.to_parquet(met_path, index=False)
print(f'Métricas salvas: {met_path}')

df_pred = pd.DataFrame({'y_true': y_te.values, 'y_proba': proba})
pred_path = os.path.join(OUTPUT_MET, f'{ALGO}_{DATASET}_predicoes.parquet')
df_pred.to_parquet(pred_path, index=False)
print(f'Predições salvas: {pred_path}')

## 10. Otimização de Hiperparâmetros (GridSearchCV)
Busca sobre `max_depth`, `min_samples_leaf`, `min_samples_split` e `criterion`.
Avaliação por `StratifiedKFold(5)` dentro do train set (2020–2023). Scoring: `average_precision` (AUPRC).

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def build_pipeline():
    return Pipeline([
        ('pre', preprocessor),
        ('clf', DecisionTreeClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE,
        )),
    ])

param_grid = {
    'clf__max_depth':        [5, 10, 20, None],
    'clf__min_samples_leaf': [5, 10, 20],
    'clf__criterion':        ['gini', 'entropy'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=build_pipeline(),
    param_grid=param_grid,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid_search.fit(X_train_prep, y_train)

print(f'\nMelhores parâmetros: {grid_search.best_params_}')
print(f'Melhor AUPRC (CV):   {grid_search.best_score_:.4f}')

df_cv = pd.DataFrame(grid_search.cv_results_)
df_cv = df_cv[['param_clf__max_depth', 'param_clf__min_samples_leaf', 'param_clf__criterion',
               'mean_test_score', 'std_test_score', 'rank_test_score']]
df_cv = df_cv.sort_values('rank_test_score')
display(df_cv)

## 11. Avaliação — Modelo Tunado

In [ ]:
pipeline_tuned = grid_search.best_estimator_

proba_tuned = pipeline_tuned.predict_proba(X_test_prep)[:, 1]
proba_tuned = proba_tuned[y_test.notna().values]

metricas_tuned = calcular_metricas(y_te, proba_tuned)

print('=== Comparação: Baseline vs Tunado ===')
print(f"{'Métrica':<18} {'Baseline':>10} {'Tunado':>10}")
print('-' * 40)
for k in ['sensibilidade', 'especificidade', 'auprc', 'roc_auc', 'f1']:
    v_base  = metricas[k]
    v_tuned = metricas_tuned[k]
    diff    = v_tuned - v_base
    sinal   = '+' if diff >= 0 else ''
    print(f'{k:<18} {v_base:>10.4f} {v_tuned:>10.4f}  ({sinal}{diff:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_te, proba,       ax=axes[0], name='Baseline')
RocCurveDisplay.from_predictions(y_te, proba_tuned, ax=axes[0], name='Tunado')
axes[0].set_title('Curva ROC — Baseline vs Tunado')

PrecisionRecallDisplay.from_predictions(y_te, proba,       ax=axes[1], name='Baseline')
PrecisionRecallDisplay.from_predictions(y_te, proba_tuned, ax=axes[1], name='Tunado')
axes[1].set_title('Curva Precision-Recall — Baseline vs Tunado')
plt.tight_layout()
plt.show()

## 12. Salvamento — Modelo Tunado

In [ ]:
DATASET_TUNED = f'{DATASET}_tuned'

model_path_tuned = os.path.join(OUTPUT_MOD, f'{ALGO}_{DATASET_TUNED}.joblib')
joblib.dump(pipeline_tuned, model_path_tuned)
print(f'Modelo salvo: {model_path_tuned}')

df_met_tuned = pd.DataFrame([metricas_tuned])
df_met_tuned['label']       = f'{ALGO}_{DATASET_TUNED}'
df_met_tuned['dataset']     = DATASET_TUNED
df_met_tuned['best_params'] = str(grid_search.best_params_)
met_path_tuned = os.path.join(OUTPUT_MET, f'{ALGO}_{DATASET_TUNED}.parquet')
df_met_tuned.to_parquet(met_path_tuned, index=False)
print(f'Métricas salvas: {met_path_tuned}')

df_pred_tuned = pd.DataFrame({'y_true': y_te.values, 'y_proba': proba_tuned})
pred_path_tuned = os.path.join(OUTPUT_MET, f'{ALGO}_{DATASET_TUNED}_predicoes.parquet')
df_pred_tuned.to_parquet(pred_path_tuned, index=False)
print(f'Predições salvas: {pred_path_tuned}')